# 🎓 U.S. Education System: Exploratory Data Analysis & Visual Analytics
**Author:** Samuel Oluolamide Adebanjo  
**Portfolio Repository:** [Samuel-Adebanjo-Data-Science-Portfolio](https://github.com/Oluola4002/Samuel-Adebanjo-Data-Science-Portfolio)  
**Domain:** Educational Data Science, Socioeconomic Equity & Institutional Performance  

---

## 📌 Project Overview & Objectives
This project delivers an in-depth exploratory data analysis (EDA) of **5,000 U.S. educational institutions** spanning primary, secondary, and post-secondary tiers.

### Key Research Questions:
1. **Institutional Demographics & Locale Distribution:** How are institutions distributed across governance controls (public, charter, private) and NCES locales (urban, suburban, rural)?
2. **Socioeconomic Impact on Proficiency:** How strongly does socioeconomic status (Free/Reduced Lunch eligibility and Title I status) correlate with math and reading performance?
3. **Finance & Educational Return:** Does higher per-pupil spending directly correlate with improved graduation rates and lower chronic absenteeism?
4. **STEM Programs & College Readiness:** What quantifiable advantage do STEM-certified high schools exhibit in SAT/ACT benchmarks and postsecondary enrollment?
5. **Digital Equity:** Does community broadband connectivity correlate with institutional academic success?


## 1. ⚙️ Environment Setup & Library Imports
We import standard scientific computing and visualization libraries, configuring styling parameters for clean, publication-ready figures.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Formatting and warning suppression
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 2)

# Publication plot aesthetics
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 10
sns.set_theme(style="whitegrid", palette="deep")
print("Environment initialized successfully!")


## 2. 📥 Resilient Data Ingestion
To support diverse runtime environments (Local IDE, JupyterLab, and Google Colab), we employ dynamic relative path resolution.

In [ ]:
# Dynamic path resolution across environments
candidate_paths = [
    "../Data/us_education_system_dataset.csv",          # Inside Portfolio/Notebook/
    "Portfolio/Data/us_education_system_dataset.csv",  # From repo root
    "Data/us_education_system_dataset.csv",            # Standard layout
    "./us_education_system_dataset.csv",               # Same directory
    "/content/drive/MyDrive/Data/us_education_system_dataset.csv" # Google Drive / Colab
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Dataset us_education_system_dataset.csv not found in candidate paths.")

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Dataset Shape: {df.shape[0]:,} rows | {df.shape[1]} columns")
df.head(5)


## 3. 🔍 Data Hygiene, Types & Missing Value Audit
We audit missing values and review structural characteristics across features.

In [ ]:
# Column overview
df.info()


In [ ]:
# Audit missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Percentage": missing_pct
}).loc[lambda x: x["Missing_Count"] > 0].sort_values(by="Missing_Count", ascending=False)

print("--- Missing Values Summary ---")
display(missing_summary)
print("Note: Null values in SAT/ACT and postsecondary metrics are expected, as they pertain only to High Schools.")


In [ ]:
# Summary statistics of key numeric metrics
core_metrics = [
    "total_enrollment", "student_teacher_ratio", "percent_free_reduced_lunch",
    "per_pupil_expenditure_usd", "math_proficiency_pct", "reading_proficiency_pct",
    "graduation_rate_pct", "chronic_absenteeism_pct", "broadband_access_pct"
]
df[core_metrics].describe().T


## 4. 🏛️ Institutional Landscape & Locale Distribution
Visualizing how educational institutions are categorized across schooling tiers, governance models, and geographic locales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution by Educational Level
level_counts = df["level"].value_counts()
sns.barplot(x=level_counts.values, y=level_counts.index, palette="mako", ax=axes[0])
axes[0].set_title("Distribution of Institutions by Educational Level", fontsize=13, fontweight="bold", pad=12)
axes[0].set_xlabel("Number of Institutions")
for i, v in enumerate(level_counts.values):
    axes[0].text(v + 15, i, f"{v:,} ({v/len(df)*100:.1f}%)", va="center", fontweight="bold", fontsize=9)

# Distribution by Institutional Control
control_counts = df["control"].value_counts()
sns.barplot(x=control_counts.values, y=control_counts.index, palette="crest", ax=axes[1])
axes[1].set_title("Institutional Control Structure", fontsize=13, fontweight="bold", pad=12)
axes[1].set_xlabel("Number of Institutions")
for i, v in enumerate(control_counts.values):
    axes[1].text(v + 25, i, f"{v:,} ({v/len(df)*100:.1f}%)", va="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# NCES Urbanicity Breakdown
plt.figure(figsize=(12, 5))
locale_order = df["nces_locale"].value_counts().index
ax = sns.countplot(data=df, y="nces_locale", order=locale_order, palette="viridis")
plt.title("Institutional Distribution Across NCES Urbanicity Locales", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Institution Count")
plt.ylabel("Locale Category")
for p in ax.patches:
    width = p.get_width()
    ax.text(width + 15, p.get_y() + p.get_height()/2, f"{int(width):,} ({width/len(df)*100:.1f}%)", va="center", fontsize=9)
plt.xlim(0, df["nces_locale"].value_counts().max() * 1.18)
plt.tight_layout()
plt.show()


## 5. 📉 Socioeconomic Equity & Academic Performance
Examining how poverty markers (Free/Reduced Lunch eligibility) and Title I federal targeting interact with academic proficiency.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Free/Reduced Lunch vs Math Proficiency
sns.regplot(
    data=df,
    x="percent_free_reduced_lunch",
    y="math_proficiency_pct",
    scatter_kws={"alpha": 0.35, "color": "#1f77b4", "s": 22},
    line_kws={"color": "#d62728", "linewidth": 2.5},
    ax=axes[0]
)
r_math, _ = stats.pearsonr(df["percent_free_reduced_lunch"], df["math_proficiency_pct"])
axes[0].set_title(f"Math Proficiency vs. Free/Reduced Lunch Rate (r = {r_math:.2f})", fontsize=12, fontweight="bold")
axes[0].set_xlabel("% Students Eligible for Free/Reduced Lunch")
axes[0].set_ylabel("Math Proficiency (%)")

# Free/Reduced Lunch vs Reading Proficiency
sns.regplot(
    data=df,
    x="percent_free_reduced_lunch",
    y="reading_proficiency_pct",
    scatter_kws={"alpha": 0.35, "color": "#2ca02c", "s": 22},
    line_kws={"color": "#d62728", "linewidth": 2.5},
    ax=axes[1]
)
r_read, _ = stats.pearsonr(df["percent_free_reduced_lunch"], df["reading_proficiency_pct"])
axes[1].set_title(f"Reading Proficiency vs. Free/Reduced Lunch Rate (r = {r_read:.2f})", fontsize=12, fontweight="bold")
axes[1].set_xlabel("% Students Eligible for Free/Reduced Lunch")
axes[1].set_ylabel("Reading Proficiency (%)")

plt.tight_layout()
plt.show()


In [ ]:
# Title I Status vs Academic Proficiency
plt.figure(figsize=(12, 6))
title_i_order = ["Title I Schoolwide", "Title I Targeted", "Not Title I Eligible", "Not Applicable"]
sns.boxplot(
    data=df,
    x="title_i_status",
    y="math_proficiency_pct",
    order=title_i_order,
    palette="Set2",
    boxprops={"alpha": 0.8},
    showmeans=True,
    meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black"}
)
plt.title("Math Proficiency by Title I Program Status", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Title I Program Status")
plt.ylabel("Math Proficiency (%)")
plt.tight_layout()
plt.show()


## 6. 💰 School Finance, Funding Sources & Expenditure ROI
Analyzing per-pupil spending patterns across institutional control models, funding composition, and correlation with graduation rates.

In [ ]:
# Per-Pupil Spending Distribution
plt.figure(figsize=(12, 6))
sns.violinplot(
    data=df,
    x="control",
    y="per_pupil_expenditure_usd",
    palette="Blues_r",
    cut=0,
    inner="quartile"
)
plt.title("Per-Pupil Expenditure ($ USD) by Institutional Control", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Institutional Control Type")
plt.ylabel("Expenditure Per Pupil ($ USD)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# Spending vs. Graduation Rate & Student-Teacher Ratio vs Absenteeism
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Spending vs Graduation Rate
sns.scatterplot(
    data=df,
    x="per_pupil_expenditure_usd",
    y="graduation_rate_pct",
    hue="control",
    alpha=0.6,
    s=30,
    ax=axes[0]
)
axes[0].set_title("Per-Pupil Expenditure vs. Graduation Rate", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Per-Pupil Expenditure ($ USD)")
axes[0].set_ylabel("Graduation Rate (%)")
axes[0].legend(loc="lower right", frameon=True)

# Student-Teacher Ratio vs Chronic Absenteeism
sns.regplot(
    data=df,
    x="student_teacher_ratio",
    y="chronic_absenteeism_pct",
    scatter_kws={"alpha": 0.35, "color": "#9467bd", "s": 22},
    line_kws={"color": "#d62728", "linewidth": 2.5},
    ax=axes[1]
)
r_abs, _ = stats.pearsonr(df["student_teacher_ratio"], df["chronic_absenteeism_pct"])
axes[1].set_title(f"Student-Teacher Ratio vs. Chronic Absenteeism (r = {r_abs:.2f})", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Student-Teacher Ratio")
axes[1].set_ylabel("Chronic Absenteeism (%)")

plt.tight_layout()
plt.show()


In [ ]:
# Funding source composition by control
funding_sources = ["federal_funding_pct", "state_funding_pct", "local_funding_pct"]
funding_by_control = df.groupby("control")[funding_sources].mean()

funding_by_control.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    colormap="tab10",
    edgecolor="white"
)
plt.title("Average Funding Revenue Composition by Institutional Control", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Institutional Control")
plt.ylabel("Share of Total Funding (%)")
plt.legend(["Federal Funding", "State Funding", "Local Funding"], loc="upper right", frameon=True)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 7. 🚀 STEM Certification, Digital Access & College Readiness
Evaluating the quantitative impact of STEM certification on standardized testing (SAT/ACT) and post-secondary transitions.

In [ ]:
# High School subset analysis
hs_df = df[df["level"] == "High School"].copy()
for col in ["avg_sat_score", "avg_act_score", "ap_ib_participation_pct", "postsecondary_enrollment_pct"]:
    hs_df[col] = pd.to_numeric(hs_df[col], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# SAT Score by STEM Certification
sns.boxplot(
    data=hs_df,
    x="stem_certified_flag",
    y="avg_sat_score",
    palette="coolwarm",
    boxprops={"alpha": 0.8},
    showmeans=True,
    meanprops={"marker": "D", "markerfacecolor": "yellow", "markeredgecolor": "black"},
    ax=axes[0]
)
axes[0].set_title("High School Average SAT Score by STEM Certification", fontsize=12, fontweight="bold")
axes[0].set_xlabel("STEM Certified Program Flag")
axes[0].set_ylabel("Average SAT Score")

# Postsecondary Enrollment by STEM Certification
sns.boxplot(
    data=hs_df,
    x="stem_certified_flag",
    y="postsecondary_enrollment_pct",
    palette="coolwarm",
    boxprops={"alpha": 0.8},
    showmeans=True,
    meanprops={"marker": "D", "markerfacecolor": "yellow", "markeredgecolor": "black"},
    ax=axes[1]
)
axes[1].set_title("Postsecondary Enrollment Rate (%) by STEM Certification", fontsize=12, fontweight="bold")
axes[1].set_xlabel("STEM Certified Program Flag")
axes[1].set_ylabel("Postsecondary Enrollment (%)")

plt.tight_layout()
plt.show()


In [ ]:
# Broadband Access vs Student Math Proficiency
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df,
    x="broadband_access_pct",
    y="math_proficiency_pct",
    scatter_kws={"alpha": 0.35, "color": "#008080", "s": 22},
    line_kws={"color": "#e74c3c", "linewidth": 2.5}
)
r_bb, _ = stats.pearsonr(df["broadband_access_pct"], df["math_proficiency_pct"])
plt.title(f"Digital Connectivity: Broadband Access vs. Math Proficiency (r = {r_bb:.2f})", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Community Broadband Access Rate (%)")
plt.ylabel("Math Proficiency (%)")
plt.tight_layout()
plt.show()


## 8. 📊 Multivariate Feature Correlation Matrix
Computing Pearson correlation coefficients across demographic, financial, attendance, and academic performance variables.

In [ ]:
# Correlation Heatmap
corr_cols = [
    "total_enrollment", "student_teacher_ratio", "percent_free_reduced_lunch",
    "special_education_pct", "english_learner_pct", "per_pupil_expenditure_usd",
    "math_proficiency_pct", "reading_proficiency_pct", "graduation_rate_pct",
    "chronic_absenteeism_pct", "broadband_access_pct"
]

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(13, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title("Multivariate Correlation Heatmap: Institutional, Socioeconomic & Performance Metrics", fontsize=13, fontweight="bold", pad=15)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. 💡 Key Findings & Strategic Insights

### 1. Socioeconomic Disparities Drive the Proficiency Gap
- Free/Reduced Lunch rates exhibit a strong negative correlation with both Math and Reading proficiency scores.
- Schools operating under Title I Schoolwide assistance programs have significantly higher concentrations of economic need, necessitating targeted curriculum intervention and resource allocation.

### 2. STEM Programs Catalyze Post-Secondary Success
- STEM-certified high schools demonstrate consistently higher average SAT/ACT scores and higher post-secondary matriculation rates compared to non-certified peers.
- Investing in STEM curriculum accreditation provides measurable long-term dividends in college readiness.

### 3. Chronic Absenteeism as an Early Warning Indicator
- Chronic absenteeism is directly associated with lower graduation rates and higher student-teacher ratios.
- Institutional interventions targeting student attendance yield compounding positive outcomes across overall school metrics.

### 4. Digital Infrastructure is Core Educational Infrastructure
- Community broadband access positively correlates with academic outcomes, underscoring that digital equity remains a vital prerequisite for modern educational parity.

---
*Analysis produced by Samuel Oluolamide Adebanjo for the Data Science Portfolio.*